# 8. Regression

선형 회귀의 loss, gradient, regularization, kernel을 직접 계산합니다. 동일 split에서 baseline과 모델을 비교합니다.

핵심 셀은 NumPy를 사용합니다. 유료 API, GPU, 외부 데이터 다운로드는 필요하지 않습니다. 실제 연구 성능을 주장하지 않는 수업용 입력입니다.

# 1. 다섯 점의 Line Fit

HTML demo와 같은 입력입니다. default w=1, b=1과 least-squares 해를 비교합니다.

In [1]:
import numpy as np
x=np.arange(5,dtype=float)
y=np.array([1,2,2,4,5.])
X=np.column_stack([x,np.ones(len(x))])
w,b=np.linalg.lstsq(X,y,rcond=None)[0]
residual=y-(w*x+b)
print("w,b:",w,b,"residual:",residual.round(4))
print("Default MSE:",np.mean((y-(x+1))**2),"Fit MSE:",np.mean(residual**2))
assert np.allclose([w,b],[1,.8]) and np.isclose(np.mean(residual**2),.16)

w,b: 1.0 0.7999999999999986 residual: [ 0.2  0.2 -0.8  0.2  0.2]
Default MSE: 0.2 Fit MSE: 0.15999999999999998


## 2. 한 번의 Gradient Update

loss는 mean squared error입니다. update 방향과 학습률을 구분하세요.

In [2]:
w0,b0=1.,1.
error=w0*x+b0-y
dw=2*np.mean(x*error);db=2*np.mean(error)
w1,b1=w0-.1*dw,b0-.1*db
print("Gradient:",dw,db,"Updated:",w1,b1)
assert np.allclose([dw,db,w1,b1],[.8,.4,.92,.96])

Gradient: 0.8 0.4 Updated: 0.9199999999999999 0.96


## 3. Ridge의 Shrinkage

아래는 sum squared loss + lambda*w² convention입니다. 앞 mean loss와 lambda의 scale을 혼동하지 마세요. centered 데이터라 intercept는 0입니다.

In [3]:
centered=np.array([-2,-1,0,1,2.])
for penalty in [0.,1.,10.,100.]:
    slope=np.dot(centered,centered)/(np.dot(centered,centered)+penalty)
    print("lambda",penalty,"slope",round(slope,4))
assert np.dot(centered,centered)==10

lambda 0.0 slope 1.0
lambda 1.0 slope 0.9091
lambda 10.0 slope 0.5
lambda 100.0 slope 0.0909


## 4. 고정된 Holdout에서 비교

seed 7로 만든 수업용 곡선과 noise입니다. 모든 모델이 같은 train/test를 씁니다. test를 보고 hyperparameter를 고르지 않습니다.

In [4]:
rng=np.random.default_rng(7)
x_all=np.linspace(-3,3,60)
y_all=np.sin(x_all)+rng.normal(0,.12,len(x_all))
indices=rng.permutation(len(x_all))
train_idx,test_idx=indices[:45],indices[45:]
xt,yt=x_all[train_idx],y_all[train_idx]
xv,yv=x_all[test_idx],y_all[test_idx]
mean_pred=np.repeat(yt.mean(),len(yv))
A=np.column_stack([xt,np.ones(len(xt))])
coef=np.linalg.lstsq(A,yt,rcond=None)[0]
linear_pred=np.column_stack([xv,np.ones(len(xv))])@coef
def metrics(pred):
    return {"MAE":float(np.mean(abs(yv-pred))),"RMSE":float(np.sqrt(np.mean((yv-pred)**2)))}
print("Mean:",metrics(mean_pred))
print("Linear:",metrics(linear_pred))

Mean: {'MAE': 0.7196730635806802, 'RMSE': 0.7642154178222319}
Linear: {'MAE': 0.4539879086844924, 'RMSE': 0.479287574884999}


In [5]:
mu=xt.mean();sd=xt.std()
z=(xt-mu)/sd;zv=(xv-mu)/sd
poly=np.column_stack([z,z*z,z*z*z])
poly_test=np.column_stack([zv,zv*zv,zv*zv*zv])
feature_mean=poly.mean(axis=0)
Pc=poly-feature_mean
target_mean=yt.mean()
coef=np.linalg.solve(Pc.T@Pc+np.eye(3),Pc.T@(yt-target_mean))
ridge_pred=(poly_test-feature_mean)@coef+target_mean
print("Cubic Ridge, sum-loss lambda1:",metrics(ridge_pred))

Cubic Ridge, sum-loss lambda1: {'MAE': 0.12659448908073373, 'RMSE': 0.166055494463226}


In [6]:
def rbf(a,b,gamma=.8):
    return np.exp(-gamma*(a[:,None]-b[None,:])**2)
K=rbf(xt,xt)
alpha=np.linalg.solve(K+.1*np.eye(len(xt)),yt)
kernel_pred=rbf(xv,xt)@alpha
print("Kernel Ridge, gamma.8 lambda.1:",metrics(kernel_pred))
assert np.all(np.isfinite(kernel_pred))
assert np.allclose(K,K.T) and np.allclose(np.diag(K),1)

Kernel Ridge, gamma.8 lambda.1: {'MAE': 0.1117852527231739, 'RMSE': 0.12989231435057738}


## 5. 과제와 해답

- original y의 한 값을 100으로 바꾸고 least-squares line의 변화를 계산하세요.
- target과 prediction을 10배하면 MAE, RMSE, MSE가 각각 몇 배가 되는지 확인하세요.

In [7]:
changed=y.copy();changed[2]=100
changed_coef=np.linalg.lstsq(X,changed,rcond=None)[0]
print("Original coefficient:",[w,b],"After outlier:",changed_coef)
e=y-(w*x+b)
print("MAE scale:",np.mean(abs(10*e))/np.mean(abs(e)))
print("RMSE scale:",np.sqrt(np.mean((10*e)**2))/np.sqrt(np.mean(e**2)))
print("MSE scale:",np.mean((10*e)**2)/np.mean(e**2))
assert np.isclose(np.mean((10*e)**2)/np.mean(e**2),100)

Original coefficient: [np.float64(1.0), np.float64(0.7999999999999986)] After outlier: [ 1.  20.4]
MAE scale: 9.999999999999998
RMSE scale: 9.999999999999998
MSE scale: 99.99999999999999


## 6. 선택: Scikit-Learn Interface

패키지가 있으면 실제 estimator로 같은 입력을 계산합니다. 미설치 환경에서도 위 핵심 실습은 이미 끝났습니다.

In [8]:
import importlib.util
if importlib.util.find_spec("sklearn") is not None:
    from sklearn.linear_model import LinearRegression
    model=LinearRegression().fit(x[:,None],y)
    assert np.allclose(model.predict(x[:,None]),w*x+b)
    print("sklearn fit matches NumPy least squares")
else:
    print("Optional sklearn cell skipped; NumPy core completed")

Optional sklearn cell skipped; NumPy core completed
